# Predictor5_0 — 通用多 AP LSTM 信号强度预测器

**核心改进：**
- 使用全部 AP 的真实数据（meme_clean.parquet）
- AP 名称通过 LabelEncoder 编码作为特征
- 时间特征（hour, day_of_week）用 sin/cos 编码
- 预测接口：`predict(ap_name, start_datetime, horizon_hours)`

**训练流程：**
1. 加载 → 2. 特征工程（AP编码+时间特征） → 3. 填充+重采样 → 4. 归一化 → 5. 序列生成 → 6. LSTM训练 → 7. 保存模型

In [ ]:
# 安装依赖（如未安装）
#!pip install pyarrow tensorflow scikit-learn matplotlib seaborn pandas joblib tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential, load_model
from keras.layers import LSTM, Dense, Dropout, Input
from keras.callbacks import EarlyStopping
from tqdm import tqdm
import warnings, os, joblib
warnings.filterwarnings('ignore')

In [ ]:
# ════════════════════════════════════════
# 配置参数
# ════════════════════════════════════════
WINDOW_SIZE      = 24       # 输入：过去 24 小时
FORECAST_HORIZON = 12       # 输出：未来 12 小时
TARGET_COLUMN    = 'signal_score'
MIN_ROWS_PER_AP  = 5000     # 最少行数（否则跳过）
BATCH_SIZE       = 64
EPOCHS           = 30
PATIENCE         = 5
MODEL_DIR        = 'predictor5_model'
os.makedirs(MODEL_DIR, exist_ok=True)

NUMERIC_FEATURES = [
    'signal_score', 'signal_strength', 'signal_db', 'snr',
    'cpu_utilization', 'mem_usage', 'client_count', 'health',
    'speed', 'maxspeed'
]
ALL_FEATURES = NUMERIC_FEATURES + ['ap_code', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
TARGET_IDX = NUMERIC_FEATURES.index(TARGET_COLUMN)  # 0

print(f'特征维度: {len(ALL_FEATURES)}')

---
## 1. 加载数据

In [ ]:
df = pd.read_parquet('meme_clean.parquet')
print(f'Shape: {df.shape}')

# 去空
df = df.dropna(subset=NUMERIC_FEATURES).copy()
print(f'去空后: {df.shape}')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['associated_device_name', 'timestamp']).reset_index(drop=True)

# 统计 AP 数据量
ap_counts = df['associated_device_name'].value_counts()
keep_aps = ap_counts[ap_counts >= MIN_ROWS_PER_AP].index
print(f'总 AP: {len(ap_counts)}, 保留(≥{MIN_ROWS_PER_AP}行): {len(keep_aps)}')

---
## 2. 特征工程

In [ ]:
# 2a. AP LabelEncoder
le_ap = LabelEncoder()
le_ap.fit(df['associated_device_name'])
df = df[df['associated_device_name'].isin(keep_aps)].copy()
df['ap_code'] = le_ap.transform(df['associated_device_name'])

# 2b. 时间特征（sin/cos 编码）
hours = df['timestamp'].dt.hour
dow   = df['timestamp'].dt.dayofweek
df['hour_sin'] = np.sin(2 * np.pi * hours / 24)
df['hour_cos'] = np.cos(2 * np.pi * hours / 24)
df['day_sin']  = np.sin(2 * np.pi * dow / 7)
df['day_cos']  = np.cos(2 * np.pi * dow / 7)

print(f'最终 Shape: {df.shape}, AP 数: {df["associated_device_name"].nunique()}')

---
## 3. 前向填充 + 重采样

In [ ]:
# 前向填充
for ap in tqdm(keep_aps, desc='前向填充'):
    mask = df['associated_device_name'] == ap
    df.loc[mask, NUMERIC_FEATURES] = df.loc[mask, NUMERIC_FEATURES].ffill().bfill()

# 按小时重采样
hourly_parts = []
for ap in tqdm(keep_aps, desc='重采样'):
    sub = df[df['associated_device_name'] == ap].copy().set_index('timestamp')
    numeric_resampled = sub[NUMERIC_FEATURES].resample('1h').mean()
    const_resampled = sub[['ap_code']].resample('1h').first()
    resampled = numeric_resampled.join(const_resampled).dropna()
    # 重新生成时间特征
    resampled['hour_sin'] = np.sin(2 * np.pi * resampled.index.hour / 24)
    resampled['hour_cos'] = np.cos(2 * np.pi * resampled.index.hour / 24)
    resampled['day_sin']  = np.sin(2 * np.pi * resampled.index.dayofweek / 7)
    resampled['day_cos']  = np.cos(2 * np.pi * resampled.index.dayofweek / 7)
    resampled['associated_device_name'] = ap
    hourly_parts.append(resampled.reset_index())

hourly_df = pd.concat(hourly_parts, ignore_index=True)
print(f'重采样后: {hourly_df.shape}')
del df

---
## 4. 归一化

In [ ]:
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(hourly_df[ALL_FEATURES])
print(f'归一化 Shape: {scaled_values.shape}')

---
## 5. 为每个 AP 生成序列 → 合并

In [ ]:
def create_sequences_for_ap(data_2d, window, horizon, target_idx):
    X, y = [], []
    total = len(data_2d) - window - horizon
    if total <= 0:
        return None, None
    for i in range(total):
        X.append(data_2d[i:i + window])
        y.append(data_2d[i + window:i + window + horizon, target_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

all_X, all_y = [], []
ap_seq_counts = {}

for ap_name in tqdm(keep_aps, desc='生成序列'):
    mask = hourly_df['associated_device_name'] == ap_name
    group = hourly_df.loc[mask].sort_values('timestamp')
    if len(group) < WINDOW_SIZE + FORECAST_HORIZON + 1:
        continue
    idx_start, idx_end = group.index[0], group.index[-1] + 1
    data_2d = scaled_values[idx_start:idx_end]
    X_ap, y_ap = create_sequences_for_ap(data_2d, WINDOW_SIZE, FORECAST_HORIZON, TARGET_IDX)
    if X_ap is not None and len(X_ap) > 0:
        all_X.append(X_ap)
        all_y.append(y_ap)
        ap_seq_counts[ap_name] = len(X_ap)

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
del all_X, all_y

print(f'实际使用 AP: {len(ap_seq_counts)}')
print(f'X: {X_all.shape}, y: {y_all.shape}')
seq_counts = pd.Series(ap_seq_counts)
print(f'中位数序列/AP: {seq_counts.median():.0f}')

---
## 6. 训练通用 LSTM

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.15, shuffle=False)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

model = Sequential([
    Input(shape=(WINDOW_SIZE, len(ALL_FEATURES))),
    LSTM(80, return_sequences=True),
    Dropout(0.25),
    LSTM(48),
    Dropout(0.25),
    Dense(32, activation='relu'),
    Dense(FORECAST_HORIZON)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
results = model.evaluate(X_test, y_test, verbose=0)
print(f'测试 MSE: {results[0]:.6f}')
print(f'测试 MAE: {results[1]:.6f}')

---
## 7. 保存模型

In [ ]:
model.save(os.path.join(MODEL_DIR, 'model.keras'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.joblib'))
joblib.dump(le_ap,  os.path.join(MODEL_DIR, 'label_encoder.joblib'))

meta = {
    'window_size': WINDOW_SIZE,
    'forecast_horizon': FORECAST_HORIZON,
    'target_column': TARGET_COLUMN,
    'numeric_features': NUMERIC_FEATURES,
    'all_features': ALL_FEATURES,
    'target_idx': TARGET_IDX,
    'n_aps_used': len(ap_seq_counts),
    'n_samples': len(X_all),
}
joblib.dump(meta, os.path.join(MODEL_DIR, 'meta.joblib'))
print(f'✅ 模型保存到 {MODEL_DIR}/')

---
## 8. 预测函数

In [ ]:
def score2label(score):
    if score >= 0.95: return 'Excellent++'
    elif score >= 0.90: return 'Excellent+'
    elif score >= 0.80: return 'Excellent'
    elif score >= 0.70: return 'Good'
    elif score >= 0.50: return 'Fair'
    else: return 'Poor'

def load_predictor(model_dir=MODEL_DIR):
    model  = load_model(os.path.join(model_dir, 'model.keras'))
    scaler = joblib.load(os.path.join(model_dir, 'scaler.joblib'))
    le_ap  = joblib.load(os.path.join(model_dir, 'label_encoder.joblib'))
    meta   = joblib.load(os.path.join(model_dir, 'meta.joblib'))
    return model, scaler, le_ap, meta

def predict(model, scaler, le_ap, meta, ap_name, start_datetime, horizon_hours=None):
    """预测指定 AP 从 start_datetime 开始的未来 signal_score
    
    Parameters
    ----------
    model : keras.Model
    scaler : MinMaxScaler
    le_ap : LabelEncoder
    meta : dict
    ap_name : str — AP 名称
    start_datetime : str/datetime — 预测起始时刻
    horizon_hours : int — 预测多少小时（默认 meta['forecast_horizon']）
    
    Returns
    -------
    list[dict] — [{'offset':1, 'timestamp':..., 'score':..., 'label':...}, ...]
    """
    W = meta['window_size']
    FH = horizon_hours or meta['forecast_horizon']
    all_feats = meta['all_features']
    
    if ap_name not in le_ap.classes_:
        raise ValueError(f"AP '{ap_name}' 不在训练集中")
    ap_code = le_ap.transform([ap_name])[0]
    
    start = pd.Timestamp(start_datetime)
    window_ts = pd.date_range(end=start - pd.Timedelta(hours=1), periods=W, freq='h')
    
    feat_data = []
    for ts in window_ts:
        feat_data.append({
            'ap_code': ap_code,
            'hour_sin': np.sin(2 * np.pi * ts.hour / 24),
            'hour_cos': np.cos(2 * np.pi * ts.hour / 24),
            'day_sin':  np.sin(2 * np.pi * ts.dayofweek / 7),
            'day_cos':  np.cos(2 * np.pi * ts.dayofweek / 7),
        })
    
    # 数值特征 — 从真实数据获取
    try:
        df_hist = pd.read_parquet('meme_clean.parquet')
        df_hist = df_hist[df_hist['associated_device_name'] == ap_name].copy()
        df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
        df_hist = df_hist.sort_values('timestamp')
        for feat_name in meta['numeric_features']:
            for i, ts in enumerate(window_ts):
                nearest = df_hist.iloc[(df_hist['timestamp'] - ts).abs().argsort()[:1]]
                feat_data[i][feat_name] = nearest[feat_name].values[0] if len(nearest) > 0 else 0.5
    except:
        for feat_name in meta['numeric_features']:
            for i in range(W):
                feat_data[i][feat_name] = 0.5
    
    input_df = pd.DataFrame(feat_data)
    input_scaled = scaler.transform(input_df[all_feats])
    X_input = np.expand_dims(input_scaled, axis=0)
    
    pred_scaled = model.predict(X_input, verbose=0)[0]
    if len(pred_scaled) < FH:
        pred_scaled = np.concatenate([pred_scaled, np.full(FH - len(pred_scaled), pred_scaled[-1])])
    pred_scaled = pred_scaled[:FH]
    
    dummy = np.zeros((FH, len(all_feats)))
    dummy[:, meta['target_idx']] = pred_scaled
    pred_original = np.clip(scaler.inverse_transform(dummy)[:, meta['target_idx']], 0, 1)
    
    result = []
    for i in range(FH):
        ts = start + pd.Timedelta(hours=i + 1)
        result.append({
            'offset': i + 1,
            'timestamp': ts,
            'score': float(pred_original[i]),
            'label': score2label(pred_original[i]),
        })
    return result

print('✅ 预测函数已定义')

---
## 9. 演示预测

In [ ]:
# 加载已保存的模型
model, scaler, le_ap, meta = load_predictor()

sample_aps = list(ap_seq_counts.keys())[:3]
print(f'示例 AP: {sample_aps}')

for ap_demo in sample_aps:
    ap_data = hourly_df[hourly_df['associated_device_name'] == ap_demo]
    last_ts = ap_data['timestamp'].max()
    print(f'\n── {ap_demo} ── 从 {last_ts} 开始预测未来 12h ──')
    preds = predict(model, scaler, le_ap, meta, ap_demo, last_ts, horizon_hours=12)
    for p in preds[:6]:
        print(f'  +{p["offset"]:>2d}h  {p["timestamp"]}  →  {p["score"]:.4f}  [{p["label"]}]')
    if len(preds) > 6:
        print(f'  ... 还有 {len(preds) - 6} 个预测')

---
## 10. 使用示例（其他脚本中调用）

In [ ]:
# 在其他脚本中这样调用:
# from run_predictor5 import load_predictor, predict
# model, scaler, le_ap, meta = load_predictor()
# result = predict(model, scaler, le_ap, meta, "AP-CEDU26", "2025-07-10 18:00:00", 24)
# for r in result:
#     print(f'{r["offset"]}h  {r["timestamp"]}  score={r["score"]:.4f}  [{r["label"]}]')
print('✅ 提示：导入方式如上')

---
## 11. 测试集对比图

In [ ]:
preds_full = model.predict(X_test, verbose=0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for idx, step in enumerate([0, 1, 5, 11]):
    ax = axes[idx // 2, idx % 2]
    ax.plot(y_test[:, step], label='Real', alpha=0.8)
    ax.plot(preds_full[:, step], label='Predicted', alpha=0.8, linestyle='--')
    ax.set_title(f'Hour +{step+1}')
    ax.set_xlabel('Test Sample')
    ax.set_ylabel('Signal Score (scaled)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()